In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression


In [2]:
df = pd.read_csv("/content/dataSet.csv")


In [3]:
df = df.drop_duplicates()


In [4]:
print(df.isnull().sum())


Hours working per day                  0
Logical quotient rating                0
hackathons                             0
coding skills rating                   0
public speaking points                 0
can work long time before system?      0
self-learning capability?              0
Extra-courses did                      0
certifications                         0
workshops                              0
talenttests taken?                     0
olympiads                              0
reading and writing skills             0
memory capability score                0
Interested subjects                    0
interested career area                 0
Job/Higher Studies?                    0
Type of company want to settle in?     0
Taken inputs from seniors or elders    0
interested in games                    0
Interested Type of Books               0
Salary Range Expected                  0
In a Realtionship?                     0
Gentle or Tuff behaviour?              0
Management or Te

In [5]:
df = df.fillna(0)


In [6]:
le = LabelEncoder()

for column in df.columns:
    if df[column].dtype == 'object':
        df[column] = le.fit_transform(df[column])


In [7]:
X = df.drop("Suggested Job Role", axis=1)
y = df["Suggested Job Role"]


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [9]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)

dt_model = DecisionTreeClassifier(random_state=42)

lr_model = LogisticRegression(max_iter=1000, random_state=42)

rf_model.fit(X_train, y_train)
dt_model.fit(X_train, y_train)
lr_model.fit(X_train, y_train)


rf_pred = rf_model.predict(X_test)
dt_pred = dt_model.predict(X_test)
lr_pred = lr_model.predict(X_test)


print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))
print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))


Random Forest Accuracy: 0.273
Decision Tree Accuracy: 0.169
Logistic Regression Accuracy: 0.285


In [10]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, lr_model.predict(X_test))
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[   0  775    0    0    0    0    0]
 [   1 1140    0    0    0    0    0]
 [   0  460    0    0    0    0    0]
 [   0  235    0    0    0    0    0]
 [   0  448    0    0    0    0    0]
 [   0  468    0    0    0    0    0]
 [   0  473    0    0    0    0    0]]


In [11]:
from sklearn.metrics import classification_report

print("Classification Report:\n")
print(classification_report(y_test, lr_model.predict(X_test)))

Classification Report:

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       775
           1       0.29      1.00      0.44      1141
           2       0.00      0.00      0.00       460
           3       0.00      0.00      0.00       235
           4       0.00      0.00      0.00       448
           5       0.00      0.00      0.00       468
           6       0.00      0.00      0.00       473

    accuracy                           0.28      4000
   macro avg       0.04      0.14      0.06      4000
weighted avg       0.08      0.28      0.13      4000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
import numpy as np

print("Total Career Classes:", len(np.unique(y)))


Total Career Classes: 7


In [13]:
import numpy as np

probabilities = lr_model.predict_proba(X_test)

top_3_indices = np.argsort(probabilities, axis=1)[:, -3:]

print("Top 3 Career Predictions (encoded):")
print(top_3_indices[0])

Top 3 Career Predictions (encoded):
[6 0 1]


In [14]:
top_3_indices = np.argsort(probabilities[0])[-3:]
top_3_careers = le.inverse_transform(top_3_indices)

print("Top 3 Career Suggestions:")
for i, career in enumerate(top_3_careers[::-1]):
    percent = probabilities[0][top_3_indices[::-1][i]] * 100
    print(f"{career} - {round(percent,2)}%")

Top 3 Career Suggestions:
Computer Science - 29.34%
Analytics - 20.87%
Networking - 12.99%


In [15]:
sample_probs = probabilities[0]
best_index = np.argmax(sample_probs)

career_name = le.inverse_transform([best_index])[0]
match_percent = sample_probs[best_index] * 100

print("Best Career:", career_name)
print("Match Percentage:", round(match_percent, 2), "%")

Best Career: Computer Science
Match Percentage: 29.34 %


In [16]:
import pickle

with open("career_model.pkl", "wb") as f:
    pickle.dump(lr_model, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

# skill Gap Analysis
## Create Ideal Career–Skill Mapping

In [17]:
sample_index = X_test.index[0]

sample_scaled = X_test[0] if isinstance(X_test, np.ndarray) else X_test.iloc[0]
original_sample = df.drop("Suggested Job Role", axis=1).loc[sample_index]



In [18]:
probabilities = lr_model.predict_proba([sample_scaled])
best_index = np.argmax(probabilities)

career_name = le.inverse_transform([best_index])[0]
match_percent = probabilities[0][best_index] * 100

print("Recommended Career:", career_name)
print("Match Percentage:", round(match_percent, 2), "%")

Recommended Career: Computer Science
Match Percentage: 29.34 %


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [19]:
weak_areas = []

for feature, value in original_sample.items():

    # Rating scale (1-5)
    if feature in ["Logical quotient rating",
                   "coding skills rating",
                   "public speaking points",
                   "reading and writing skills",
                   "memory capability score"]:

        if value <= 2:
            weak_areas.append(feature)

    # Binary features
    elif feature in ["self-learning capability?",

                     "worked in teams ever?"]:

        if value == 0:
            weak_areas.append(feature)

print("\nAreas to Improve:")
for area in weak_areas:
    print("-", area)



Areas to Improve:
- coding skills rating
- public speaking points
- self-learning capability?
- reading and writing skills
- memory capability score


Action Plan Dictionary

In [20]:
action_plan_map = {
    "coding skills rating": "Practice coding daily on platforms like LeetCode or HackerRank.",
    "Logical quotient rating": "Solve aptitude and logical reasoning problems regularly.",
    "public speaking points": "Join debate clubs or practice presentations weekly.",
    "reading and writing skills": "Read technical blogs and write summaries daily.",
    "memory capability score": "Practice memory techniques and revise concepts frequently.",
    "self-learning capability?": "Take online certifications and build self-study habits.",
    "can work long time before system?": "Improve focus using Pomodoro technique.",
    "worked in teams ever?": "Participate in group projects or hackathons."
}


In [21]:
print("\n Personalized Action Plan:")

if weak_areas:
    for area in weak_areas:
        suggestion = action_plan_map.get(area, "Work on improving this area consistently.")
        print(f"• {area} → {suggestion}")
else:
    print("You already have strong alignment with this career. Keep improving consistently!")



 Personalized Action Plan:
• coding skills rating → Practice coding daily on platforms like LeetCode or HackerRank.
• public speaking points → Join debate clubs or practice presentations weekly.
• self-learning capability? → Take online certifications and build self-study habits.
• reading and writing skills → Read technical blogs and write summaries daily.
• memory capability score → Practice memory techniques and revise concepts frequently.


Hyperparameter Tuning for Logistic Regression

In [22]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['lbfgs', 'liblinear']
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)


Best Parameters: {'C': 0.01, 'solver': 'liblinear'}
Best CV Score: 0.2852499944163646


In [23]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

from sklearn.metrics import accuracy_score
print("Tuned Model Accuracy:", accuracy_score(y_test, y_pred))


Tuned Model Accuracy: 0.28525
